# 05 — Advanced RAG Patterns**Learning objectives:**- HyDE (Hypothetical Document Embedding)- Query rewriting- Reranking before generation

## Setup

In [ ]:
import sysfrom pathlib import PathROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()sys.path.insert(0, str(ROOT))from llama_index.core import Settingsfrom src.config import FALLBACK_DIR, print_configfrom src.ingestion.loader import load_documentsfrom src.ingestion.chunker import chunk_documentsfrom src.ingestion.embedder import configure_embed_modelfrom src.retrieval.store import get_or_create_indexfrom src.utils.fallback import get_llmprint_config()documents = load_documents(FALLBACK_DIR / "squad_sample.json")embed_model = configure_embed_model()nodes = chunk_documents(documents, strategy="sentence")index = get_or_create_index(nodes=nodes, embed_model=embed_model, force_rebuild=True)llm = get_llm()Settings.llm = llm

## Query Rewriting

In [ ]:
# ── EXERCISE ──────────────────────────────────────────────────────────────# Rewrite a vague query to be more retrieval-friendly.original = "Who led the revolution?"# YOUR CODE HERErewritten = ???print(f"Original: {original}")print(f"Rewritten: {rewritten}")

In [ ]:
# ── SOLUTION ──────────────────────────────────────────────────────────────from src.generation.prompts import QUERY_REWRITE_TEMPLATEoriginal = "Who led the revolution?"prompt = QUERY_REWRITE_TEMPLATE.format(query_str=original)rewritten = llm.complete(prompt).text.strip()print(f"Original: {original}")print(f"Rewritten: {rewritten}")

## HyDE + Reranking

In [ ]:
# ── EXERCISE ──────────────────────────────────────────────────────────────# Generate a hypothetical document, retrieve, and rerank.query = "Who was the first president?"# YOUR CODE HEREreranked = ???print(f"Top result: {reranked[0].node.get_content()[:200]}...")

In [ ]:
# ── SOLUTION ──────────────────────────────────────────────────────────────from src.generation.prompts import HYDE_TEMPLATEfrom src.retrieval.retriever import get_hybrid_retrieverfrom src.retrieval.reranker import build_rerankerfrom src.utils.display import print_retrieval_resultsquery = "Who was the first president?"hyde_prompt = HYDE_TEMPLATE.format(query_str=query)hypothetical = llm.complete(hyde_prompt).text.strip()retriever = get_hybrid_retriever(index, nodes, top_k=10)retrieved = retriever.retrieve(hypothetical[:500])print_retrieval_results(retrieved[:5], "Before Reranking")reranker = build_reranker(top_n=3)reranked = reranker.postprocess_nodes(retrieved, query_str=query)print_retrieval_results(reranked, "After Reranking")